### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [32]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ['GEMINI_API_KEY']=os.getenv("GOOGLE_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [33]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
# client.delete_dataset(dataset_name)
client.delete_dataset(dataset_name="Chatbots Evaluation")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['a0ac343a-edbd-4b4b-8600-7e0cdfa22045',
  'c30ba088-b139-4e78-9c9d-815c67e3647e',
  '48f60cda-f969-473f-8eb3-55bb314d5fb4',
  '923953e9-17c9-4842-9ca7-2609cb0b803b',
  '1b80e7e6-53d0-4b8c-947c-0fe9b5e1e40b'],
 'count': 5,
 'as_of': '2026-05-01T09:14:41.596614037Z'}

### Define Metrics (LLM As A Judge)


In [34]:
# import openai
from langsmith import wrappers
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage,HumanMessage

ollama_llm = ChatOllama(model="llama3.2:latest")

# ollama_wrapper=wrappers.wrap_openai(ollama_llm)


eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    #   response=openai_client.chat.completions.create(
    #         model="gpt-4o-mini",
    #         temperature=0,
    #         messages=[
    #               {"role":"system","content":eval_instructions},
    #               {"role":"user","content":user_content}
    #         ]
    #   ).choices[0].message.content
    response = ollama_llm.invoke([
        SystemMessage(content=eval_instructions),
        HumanMessage(content=user_content)
    ])

    grade = response.content.strip().upper()

    return "CORRECT" in grade

In [35]:
# def concision(output:dict, reference_output:dict) -> bool:
#     return int(len(output['response'])<2*len(reference_output['answer']))
def concision(run, example):
    prediction = run.outputs["response"]
    reference = example.outputs["answer"]
    
    return {
        "score": int(len(prediction) < 2 * len(reference)),
        "key": "concision"
    }

## Run Evaluations

In [39]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str,  instructions: str = default_instructions) -> str:
    return ollama_llm.invoke([
        SystemMessage(content=instructions),
        HumanMessage(content=question)
    ]).content.strip()

In [40]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [41]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="ollama-llama3.2latest-chatbot"
)

View the evaluation results for experiment: 'ollama-llama3.2latest-chatbot-5ba80ab4' at:
https://smith.langchain.com/o/b3fd304e-4ddf-41e0-bcfd-b348b4ec7b53/datasets/d6a5f319-141b-4830-af16-d84840c6dfb2/compare?selectedSessions=ec6c3c79-4fea-40df-929a-69b30fe6d4fc




0it [00:00, ?it/s]

In [42]:
experiment_results

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.concision,execution_time,example_id,id
0,What is Mistral?,"Mistral refers to either the French word for ""...",None,A company that creates Large Language Models,True,1,4.869178,1b80e7e6-53d0-4b8c-947c-0fe9b5e1e40b,019de340-e6b5-7d52-af89-7fff8f8603a1
1,What is OpenAI?,OpenAI is an artificial intelligence research ...,None,A company that creates Large Language Models,True,0,0.491138,48f60cda-f969-473f-8eb3-55bb314d5fb4,019de340-fb7d-7560-9638-ef95cb95fd5b
2,What is Google?,Google is a multinational technology company t...,None,A technology company known for search,True,0,0.330299,923953e9-17c9-4842-9ca7-2609cb0b803b,019de341-00c2-7350-b588-8eb941191633
3,What is LangChain?,LangChain is an open-source framework for buil...,None,A framework for building LLM applications,True,0,0.358835,a0ac343a-edbd-4b4b-8600-7e0cdfa22045,019de341-0439-7c41-a2bf-317a2086eb77
4,What is LangSmith?,Langsmith is an expert who translates Hebrew a...,None,A platform for observing and evaluating LLM ap...,True,1,0.355676,c30ba088-b139-4e78-9c9d-815c67e3647e,019de341-090a-75a1-a85a-918702befd01
